# OpenReview Acceptance Predictor: Colab Pipeline

This notebook is designed to run sequentially from a prebuilt processed dataset. It installs dependencies, restores the committed compressed dataset if needed, creates splits, trains the baselines, builds retrieval, runs RAG, and optionally runs LoRA fine-tuning.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')

def box(x, y, w, h, label, color, required=True):
    style = 'round,pad=0.05,rounding_size=0.15'
    edge = 'black' if required else '#888888'
    ls = '-' if required else '--'
    p = FancyBboxPatch((x, y), w, h, boxstyle=style,
                       facecolor=color, edgecolor=edge, linewidth=1.6, linestyle=ls)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=9.5, wrap=True)

def arrow(x1, y1, x2, y2, dashed=False):
    style = '->' if not dashed else '->'
    ls = '--' if dashed else '-'
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style,
                        mutation_scale=14, color='#444', linestyle=ls, linewidth=1.4)
    ax.add_patch(a)

CORE = '#cfe8e0'
OPT  = '#fde2c4'
OUT  = '#e6e6f5'

box(0.2, 6.3, 2.4, 1.2, 'Processed dataset\npapers.jsonl.gz', CORE)
box(3.2, 6.3, 2.4, 1.2, 'Strip label leaks\n(cell 8)', CORE)
box(6.2, 6.3, 2.4, 1.2, 'Train/Dev/Test splits\n03_make_splits.py', CORE)
box(9.2, 6.3, 2.6, 1.2, 'TF-IDF + LogReg\n04_train_baselines.py', CORE)

box(9.2, 4.0, 2.6, 1.2, 'Metrics + figures\n(cells 19, 21, 22, 24, 25)', OUT)

box(0.2, 2.4, 2.4, 1.2, 'Retrieval index\n05_build_retrieval_index.py', OPT, required=False)
box(3.2, 2.4, 2.4, 1.2, 'RAG predict\n06_rag_predict.py\n(TinyLlama-1.1B)', OPT, required=False)
box(6.2, 2.4, 2.4, 1.2, 'Retrieval-only fallback\n(cell 23)', OPT, required=False)

box(0.2, 0.4, 2.4, 1.2, 'LoRA fine-tune\n07_finetune_lora.py\n(needs GPU)', OPT, required=False)

arrow(2.6, 6.9, 3.2, 6.9)
arrow(5.6, 6.9, 6.2, 6.9)
arrow(8.6, 6.9, 9.2, 6.9)
arrow(10.5, 6.3, 10.5, 5.2)

arrow(7.4, 6.3, 7.4, 3.6, dashed=True)
arrow(2.6, 3.0, 3.2, 3.0, dashed=True)
arrow(5.6, 3.0, 6.2, 3.0, dashed=True)
arrow(7.4, 2.4, 9.5, 1.6, dashed=True)
arrow(7.4, 4.0, 7.4, 3.6, dashed=True)

arrow(7.4, 6.3, 1.4, 1.6, dashed=True)

ax.text(0.2, 7.7, 'Required path (TF-IDF baseline)', fontsize=10, fontweight='bold')
ax.text(0.2, 3.8, 'Optional: Retrieval-Augmented Model (RAG)', fontsize=10, fontweight='bold', color='#7a4a00')
ax.text(0.2, 1.8, 'Optional: LoRA fine-tuning', fontsize=10, fontweight='bold', color='#7a4a00')

ax.text(11.8, 0.2, 'solid = required, dashed = optional',
        fontsize=8, style='italic', color='#555', ha='right')

plt.title('OpenReview Acceptance Predictor — Pipeline Overview', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/trinayhari/ml-paper-evaluator.git'
CLONE_DIR = '/content/ml-paper-evaluator'
%cd /content
!git clone {REPO_URL} {CLONE_DIR} || true
%cd /content/ml-paper-evaluator


In [ ]:
!python -m pip install -r requirements.txt


## Configure dataset and run options

Set `PROCESSED_DATA` to the extracted dataset you already created locally. `OUTPUT_ROOT` is where all generated splits, models, metrics, and predictions will be written.


In [ ]:
PROCESSED_DATA = '/content/ml-paper-evaluator/data/processed/papers.jsonl.gz'
OUTPUT_ROOT = '/content/drive/MyDrive/openreview/colab_pipeline_run'
SEED = 42
TEST_VENUES = []
RAG_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
RAG_LIMIT = 100
RAG_K = 5
RUN_LORA = False
LORA_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

ROOT = Path('/content/ml-paper-evaluator')
processed_path = Path(PROCESSED_DATA)
output_root = Path(OUTPUT_ROOT)
splits_dir = output_root / 'splits'
baselines_dir = output_root / 'baselines'
retrieval_dir = output_root / 'retrieval'
rag_out = output_root / 'rag_predictions.jsonl'
lora_dir = output_root / 'lora_model'

def run_cmd(cmd):
    print('\n==>', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

if not processed_path.exists():
    part_candidates = sorted(processed_path.parent.glob(processed_path.name + '.part-*'))
    if part_candidates:
        run_cmd(['python', 'scripts/00_restore_dataset.py', '--out', str(processed_path)])

assert processed_path.exists(), f'Missing processed dataset: {processed_path}'
print('Using dataset:', processed_path)


## Strip label leakage from PDF text

ICLR camera-ready PDFs carry the header *"Published as a conference paper at ICLR ..."* on every page, and submission PDFs carry *"Under review as a conference paper at ICLR ..."*. These phrases trivially reveal the accept/reject label, so TF-IDF picks them up and inflates accuracy to >90%. The cell below strips those phrases (and a few smaller leaks) and repoints `processed_path` to the cleaned file so all downstream stages use it.

In [ ]:
import gzip
import json
import re

LEAK_PATTERNS = [
    re.compile(r'published as a conference paper at iclr[^\n]*', re.IGNORECASE),
    re.compile(r'under review as a conference paper at iclr[^\n]*', re.IGNORECASE),
    re.compile(r'we thank the (?:anonymous )?reviewers[^\n]*', re.IGNORECASE),
    re.compile(r'meta-?review[^\n]*', re.IGNORECASE),
]

def clean_text(t):
    if not t:
        return t
    for pat in LEAK_PATTERNS:
        t = pat.sub(' ', t)
    return t

clean_path = processed_path.with_name('papers_clean.jsonl.gz')
in_open = gzip.open if processed_path.suffix == '.gz' else open
out_open = gzip.open if clean_path.suffix == '.gz' else open

n = 0
hits_before = 0
with in_open(processed_path, 'rt') as fin, out_open(clean_path, 'wt') as fout:
    for line in fin:
        r = json.loads(line)
        for field in ('title', 'abstract', 'paper_text'):
            if field in r and r[field]:
                original = r[field]
                cleaned = clean_text(original)
                if cleaned != original:
                    hits_before += 1
                r[field] = cleaned
        fout.write(json.dumps(r) + '\n')
        n += 1

processed_path = clean_path
print(f'wrote {n} rows to {clean_path}')
print(f'leak phrases stripped from {hits_before} field instances')
print(f'processed_path now points to: {processed_path}')

## Option A: run the whole pipeline in one command

This cell is the easiest path for someone who just wants to execute everything after setting the dataset path above.


In [ ]:
if output_root.exists():
    shutil.rmtree(output_root)

cmd = [
    'python', 'scripts/08_run_colab_pipeline.py',
    '--input', str(processed_path),
    '--output-root', str(output_root),
    '--seed', str(SEED),
    '--clean-output',
    '--rag-model', RAG_MODEL,
    '--rag-limit', str(RAG_LIMIT),
    '--rag-k', str(RAG_K),
]
if TEST_VENUES:
    cmd.extend(['--test-venues', *TEST_VENUES])
if RUN_LORA:
    cmd.extend(['--run-lora', '--lora-model', LORA_MODEL])
run_cmd(cmd)


## Option B: run each stage explicitly

Use the next cells if you want to see each stage run separately. They use the same dataset and output paths configured above.


In [ ]:
split_cmd = [
    'python', 'scripts/03_make_splits.py',
    '--input', str(processed_path),
    '--out-dir', str(splits_dir),
    '--seed', str(SEED),
]
if TEST_VENUES:
    split_cmd.extend(['--test-venues', *TEST_VENUES])
run_cmd(split_cmd)


In [ ]:
run_cmd([
    'python', 'scripts/04_train_baselines.py',
    '--train', str(splits_dir / 'train.jsonl'),
    '--dev', str(splits_dir / 'dev.jsonl'),
    '--test', str(splits_dir / 'test.jsonl'),
    '--out', str(baselines_dir),
])


In [ ]:
run_cmd([
    'python', 'scripts/05_build_retrieval_index.py',
    '--train', str(splits_dir / 'train.jsonl'),
    '--out', str(retrieval_dir),
])


In [ ]:
run_cmd([
    'python', 'scripts/06_rag_predict.py',
    '--test', str(splits_dir / 'test.jsonl'),
    '--index-dir', str(retrieval_dir),
    '--out', str(rag_out),
    '--model', RAG_MODEL,
    '--limit', str(RAG_LIMIT),
    '--k', str(RAG_K),
])


## Optional LoRA fine-tuning

Run this only if your Colab runtime has a GPU with enough memory and you intentionally want the fine-tuning step.


In [ ]:
if RUN_LORA:
    run_cmd([
        'python', 'scripts/07_finetune_lora.py',
        '--train', str(splits_dir / 'train.jsonl'),
        '--dev', str(splits_dir / 'dev.jsonl'),
        '--model', LORA_MODEL,
        '--out', str(lora_dir),
    ])
else:
    print('Skipping LoRA because RUN_LORA is False')


In [ ]:
summary_path = output_root / 'run_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    summary
else:
    print('No run_summary.json found. If you ran stages manually, inspect the output folders directly.')
    print('Output root:', output_root)


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

metrics_path = Path(OUTPUT_ROOT) / 'baselines' / 'metrics.json'
assert metrics_path.exists(), f'Run cell 13 first to produce {metrics_path}'

with open(metrics_path) as f:
    metrics = json.load(f)

grouped = defaultdict(dict)
for key, vals in metrics.items():
    if key.endswith('_train'):
        grouped[key[:-6]]['train'] = vals
    elif key.endswith('_dev'):
        grouped[key[:-4]]['dev'] = vals
    elif key.endswith('_test'):
        grouped[key[:-5]]['test'] = vals

cols = ['accuracy', 'f1', 'auroc', 'brier', 'ece']
splits = ['train', 'dev', 'test']

header = f'{"model":24s}  {"split":6s}  ' + '  '.join(f'{c:>8s}' for c in cols)
print(header)
print('-' * len(header))
for model in sorted(grouped):
    for split in splits:
        m = grouped[model].get(split)
        if not m:
            continue
        row = '  '.join(f'{m.get(c, float("nan")):8.4f}' for c in cols)
        print(f'{model:24s}  {split:6s}  {row}')
    tr, te = grouped[model].get('train'), grouped[model].get('test')
    if tr and te:
        gap = tr.get('accuracy', 0) - te.get('accuracy', 0)
        print(f'{"":24s}  {"gap":6s}  ' + ' ' * 8 +
              f'  train_acc - test_acc = {gap:+.4f}'
              + ('   <-- possible overfitting' if gap > 0.10 else ''))
    print()

In [ ]:
metrics_path = baselines_dir / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics
else:
    print('Missing metrics file:', metrics_path)


# Analysis & figures for the report

The cells below run after the pipeline finishes. They produce:

1. Dataset statistics, majority-class baseline, and leak re-check (sanity proof the cleaning worked).
2. Top TF-IDF features per class (Figure 1 — visual proof the leak is gone).
3. Retrieval-only RAG fallback (real, varied probabilities for the RAG row in Table 1).
4. Calibration curve for TF-IDF.
5. Error analysis: 5 misclassified test papers with their abstracts.

In [ ]:
import json
import collections
from pathlib import Path

OUT = Path(OUTPUT_ROOT)
splits = OUT / 'splits'

print('=' * 60)
print('Dataset statistics')
print('=' * 60)
for name in ('train', 'dev', 'test'):
    rows = [json.loads(l) for l in open(splits / f'{name}.jsonl')]
    accept = sum(1 for r in rows if int(r['label']) == 1)
    venues = collections.Counter(r.get('venue', '?') for r in rows)
    print(f'{name:6s}  n={len(rows):5d}  accept={accept:5d} ({accept/len(rows):.1%})  '
          f'reject={len(rows)-accept:5d} ({(len(rows)-accept)/len(rows):.1%})')
    print(f'        venues: {dict(venues)}')

test_rows = [json.loads(l) for l in open(splits / 'test.jsonl')]
test_accept_rate = sum(1 for r in test_rows if int(r['label']) == 1) / len(test_rows)
majority_acc = max(test_accept_rate, 1 - test_accept_rate)

print('\n' + '=' * 60)
print('Majority-class baseline')
print('=' * 60)
print(f'  test accept rate     : {test_accept_rate:.3f}')
print(f'  majority-class acc   : {majority_acc:.3f}')
print(f'  majority-class AUROC : 0.500 (no ranking ability)')

print('\n' + '=' * 60)
print('Leak re-check on cleaned training data')
print('=' * 60)
leak_terms = [
    'published as a conference paper',
    'under review as a conference paper',
    'we thank the reviewers',
    'meta-review',
]
by_label = {0: collections.Counter(), 1: collections.Counter()}
for line in open(splits / 'train.jsonl'):
    r = json.loads(line)
    text = (r.get('paper_text', '') or '').lower()
    for t in leak_terms:
        if t in text:
            by_label[int(r['label'])][t] += 1
for t in leak_terms:
    print(f'  {t:42s}  reject={by_label[0][t]:5d}  accept={by_label[1][t]:5d}')
print('\nIf the top two phrases show 0/0 in both classes, the cleaning succeeded.')

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path(OUTPUT_ROOT)
pipe = joblib.load(OUT / 'baselines' / 'tfidf_logreg.joblib')

vec = pipe.named_steps['tfidf']
clf = pipe.named_steps['clf']
feats = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]

n_top = 15
top_accept_idx = np.argsort(coef)[-n_top:][::-1]
top_reject_idx = np.argsort(coef)[:n_top]

print('Top features predicting ACCEPT:')
for i in top_accept_idx:
    print(f'  {coef[i]:+.3f}  {feats[i]}')
print('\nTop features predicting REJECT:')
for i in top_reject_idx:
    print(f'  {coef[i]:+.3f}  {feats[i]}')

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=False)
axes[0].barh(range(n_top), coef[top_accept_idx][::-1], color='#2a9d8f')
axes[0].set_yticks(range(n_top))
axes[0].set_yticklabels(feats[top_accept_idx][::-1], fontsize=9)
axes[0].set_title('Top features predicting ACCEPT')
axes[0].set_xlabel('Logistic regression coefficient')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(range(n_top), coef[top_reject_idx][::-1], color='#e76f51')
axes[1].set_yticks(range(n_top))
axes[1].set_yticklabels(feats[top_reject_idx][::-1], fontsize=9)
axes[1].set_title('Top features predicting REJECT')
axes[1].set_xlabel('Logistic regression coefficient')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('TF-IDF + Logistic Regression: Top Features After Leak Removal', fontsize=12)
plt.tight_layout()

fig_path = OUT / 'figure_top_features.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved to {fig_path}')

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss,
)

OUT = Path(OUTPUT_ROOT)
splits_dir = OUT / 'splits'
metrics_path = OUT / 'baselines' / 'metrics.json'
rag_main_path = OUT / 'rag_predictions.jsonl'
rag_fallback_path = OUT / 'rag_predictions_retrieval_only.jsonl'
fallback_metrics_path = OUT / 'rag_retrieval_only_metrics.json'

assert metrics_path.exists(), f'Run cell 13 first to produce {metrics_path}'

COLS = ['accuracy', 'precision', 'recall', 'f1', 'auroc', 'brier', 'ece']

def empty_row():
    return {c: float('nan') for c in COLS}

def metrics_from_probs(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'brier':     brier_score_loss(y_true, y_prob),
        'auroc':     roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
        'ece':       float('nan'),
    }
    return out

def load_split_labels(name):
    rows = [json.loads(l) for l in open(splits_dir / f'{name}.jsonl')]
    return np.array([int(r['label']) for r in rows])

with open(metrics_path) as f:
    raw_metrics = json.load(f)

table = defaultdict(lambda: {'dev': empty_row(), 'test': empty_row()})

PRETTY = {
    'tfidf_logreg':      'TF-IDF + LR',
    'sbert_logreg':      'NBOW (SBERT mean) + LR',
    'tfidf_embed_logreg':'NBOW fallback (TF-IDF) + LR',
}

for key, vals in raw_metrics.items():
    for split in ('dev', 'test'):
        suffix = f'_{split}'
        if key.endswith(suffix):
            model = key[:-len(suffix)]
            display = PRETTY.get(model, model)
            row = empty_row()
            for c in COLS:
                if c in vals:
                    row[c] = vals[c]
            table[display][split] = row

ydev_true = load_split_labels('dev')
yte_true  = load_split_labels('test')

dev_majority_prob = float(ydev_true.mean())
te_majority_prob  = float(yte_true.mean())
maj_dev_pred = np.full_like(ydev_true, int(dev_majority_prob >= 0.5))
maj_te_pred  = np.full_like(yte_true,  int(te_majority_prob  >= 0.5))

def majority_row(y_true, y_pred, accept_rate):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auroc':     0.5,
        'brier':     float(np.mean((y_true - accept_rate) ** 2)),
        'ece':       float('nan'),
    }

table['Majority class']['dev']  = majority_row(ydev_true, maj_dev_pred, dev_majority_prob)
table['Majority class']['test'] = majority_row(yte_true,  maj_te_pred,  te_majority_prob)

def add_rag_row(label, path):
    if not path.exists():
        return
    rows = [json.loads(l) for l in open(path)]
    if not rows:
        return
    y_true = [int(r['true_label']) for r in rows]
    y_prob = [float(r['probability_accept']) for r in rows]
    table[label]['test'] = metrics_from_probs(y_true, y_prob)

add_rag_row('RAG (LLM, k=5)',          rag_main_path)
add_rag_row('RAG (retrieval-only k=5)', rag_fallback_path)

ROW_ORDER = [
    'Majority class',
    'TF-IDF + LR',
    'NBOW (SBERT mean) + LR',
    'NBOW fallback (TF-IDF) + LR',
    'RAG (retrieval-only k=5)',
    'RAG (LLM, k=5)',
]
ordered = [r for r in ROW_ORDER if r in table] + [r for r in table if r not in ROW_ORDER]

def fmt(v):
    return '   -  ' if v is None or (isinstance(v, float) and np.isnan(v)) else f'{v:6.3f}'

print('=' * 100)
print('Table 1 — Results on dev / test (binary accept vs reject)')
print('=' * 100)
header = f'{"Model":30s}  {"Split":5s}  ' + '  '.join(f'{c:>6s}' for c in COLS)
print(header)
print('-' * len(header))
for model in ordered:
    for split in ('dev', 'test'):
        row = table[model][split]
        vals = '  '.join(fmt(row[c]) for c in COLS)
        print(f'{model:30s}  {split:5s}  {vals}')
    print()

md_lines = ['# Table 1 — Results', '']
md_lines.append('| Model | Split | ' + ' | '.join(c.upper() for c in COLS) + ' |')
md_lines.append('|' + '---|' * (len(COLS) + 2))
for model in ordered:
    for split in ('dev', 'test'):
        row = table[model][split]
        cells = ['—' if (isinstance(row[c], float) and np.isnan(row[c])) else f'{row[c]:.3f}' for c in COLS]
        md_lines.append(f'| {model} | {split} | ' + ' | '.join(cells) + ' |')

md_path = OUT / 'table1_results.md'
md_path.write_text('\n'.join(md_lines))

import csv
csv_path = OUT / 'table1_results.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'split', *COLS])
    for model in ordered:
        for split in ('dev', 'test'):
            row = table[model][split]
            w.writerow([model, split, *[row[c] for c in COLS]])

print(f'\nSaved: {md_path}')
print(f'Saved: {csv_path}')

print('\nNBOW sanity check:')
if 'sbert_logreg_test' in raw_metrics:
    print('  sentence-transformers loaded — `NBOW (SBERT mean) + LR` row is the real NBOW baseline.')
elif 'tfidf_embed_logreg_test' in raw_metrics:
    print('  WARNING: sentence-transformers not available; the NBOW slot fell back to a second TF-IDF model.')
    print('  To get the real NBOW baseline, ensure `sentence-transformers` installs in the runtime and re-run cell 13.')
else:
    print('  Could not find an NBOW-style baseline row in metrics.json.')

In [ ]:
import json
import subprocess
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    brier_score_loss, f1_score, precision_score, recall_score,
)

OUT = Path(OUTPUT_ROOT)
ROOT = Path('/content/ml-paper-evaluator')
fallback_out = OUT / 'rag_predictions_retrieval_only.jsonl'

cmd = [
    'python', 'scripts/06_rag_predict.py',
    '--test', str(OUT / 'splits' / 'test.jsonl'),
    '--index-dir', str(OUT / 'retrieval'),
    '--out', str(fallback_out),
    '--model', 'definitely-not-a-real/model-force-fallback',
    '--limit', '200',
    '--k', '5',
]
print('==>', ' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

rows = [json.loads(l) for l in open(fallback_out)]
unique = sorted({round(r['probability_accept'], 3) for r in rows})
print(f'\nn = {len(rows)}')
print(f'unique probability values ({len(unique)}): {unique}')

y_true = [int(r['true_label']) for r in rows]
y_prob = [float(r['probability_accept']) for r in rows]
y_pred = [1 if p >= 0.5 else 0 for p in y_prob]

rag_metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, zero_division=0),
    'recall': recall_score(y_true, y_pred, zero_division=0),
    'f1': f1_score(y_true, y_pred, zero_division=0),
    'brier': brier_score_loss(y_true, y_prob),
    'auroc': roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
    'n': len(rows),
}
print('\nRetrieval-only RAG (k=5) metrics on test set:')
for k, v in rag_metrics.items():
    print(f'  {k:10s} {v:.4f}' if isinstance(v, float) else f'  {k:10s} {v}')
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=['REJECT', 'ACCEPT'], zero_division=0))

with open(OUT / 'rag_retrieval_only_metrics.json', 'w') as f:
    json.dump(rag_metrics, f, indent=2)
print(f'Saved metrics to {OUT / "rag_retrieval_only_metrics.json"}')

In [ ]:
import json
import sys
import joblib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.calibration import calibration_curve

ROOT = Path('/content/ml-paper-evaluator')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.ml_utils import build_text_fields

OUT = Path(OUTPUT_ROOT)
test_rows = [json.loads(l) for l in open(OUT / 'splits' / 'test.jsonl')]
xte = build_text_fields(test_rows, max_words=3500)
yte = np.array([int(r['label']) for r in test_rows])

pipe = joblib.load(OUT / 'baselines' / 'tfidf_logreg.joblib')
y_prob = pipe.predict_proba(xte)[:, 1]

frac_pos, mean_pred = calibration_curve(yte, y_prob, n_bins=10, strategy='uniform')

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Perfectly calibrated')
plt.plot(mean_pred, frac_pos, 'o-', color='#2a9d8f', label='TF-IDF + LR (cleaned)')
plt.xlabel('Mean predicted probability of ACCEPT')
plt.ylabel('Fraction actually ACCEPTED')
plt.title('Calibration Curve: TF-IDF + LR on Test Set')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()

fig_path = OUT / 'figure_calibration.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {fig_path}')

In [ ]:
import json
import sys
import joblib
import numpy as np
from pathlib import Path

ROOT = Path('/content/ml-paper-evaluator')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.ml_utils import build_text_fields

OUT = Path(OUTPUT_ROOT)
test_rows = [json.loads(l) for l in open(OUT / 'splits' / 'test.jsonl')]
xte = build_text_fields(test_rows, max_words=3500)
yte = np.array([int(r['label']) for r in test_rows])

pipe = joblib.load(OUT / 'baselines' / 'tfidf_logreg.joblib')
probs = pipe.predict_proba(xte)[:, 1]
preds = (probs >= 0.5).astype(int)

errors = []
for r, p, prob, pred in zip(test_rows, probs, probs, preds):
    if pred != int(r['label']):
        errors.append({
            'forum': r.get('forum', ''),
            'title': r.get('title', ''),
            'abstract': r.get('abstract', ''),
            'true_label': int(r['label']),
            'pred_label': int(pred),
            'prob_accept': float(prob),
            'confidence': abs(float(prob) - 0.5),
        })

errors.sort(key=lambda e: -e['confidence'])

print(f'{len(errors)} errors out of {len(test_rows)} test papers ({len(errors)/len(test_rows):.1%})')
print('\nTop confidently-wrong predictions (worst mistakes first):\n')

selected = errors[:5]
for i, e in enumerate(selected, 1):
    true = 'ACCEPT' if e['true_label'] == 1 else 'REJECT'
    pred = 'ACCEPT' if e['pred_label'] == 1 else 'REJECT'
    err_type = 'False Positive' if e['pred_label'] == 1 else 'False Negative'
    print(f'--- Error {i}: {err_type} (true={true}, pred={pred} @ p={e["prob_accept"]:.3f}) ---')
    print(f'TITLE: {e["title"][:160]}')
    print(f'ABSTRACT: {e["abstract"][:600]}')
    print()

with open(OUT / 'error_analysis.json', 'w') as f:
    json.dump(selected, f, indent=2)
print(f'Saved {len(selected)} examples to {OUT / "error_analysis.json"}')